Event Mannagment

In [0]:
%sql
CREATE OR REPLACE TABLE event_registrations_source (
  registration_id INT, event_id STRING, attendee_name STRING, registration_date STRING,
  ticket_type STRING, tickets INT, unit_price INT, status STRING, channel STRING
) USING DELTA;

In [0]:
%sql
INSERT INTO event_registrations_source VALUES
(2001,'E01','Aarav','2026-10-01','Standard',2,500,'Confirmed','Web'),
(2002,'E02','Arnav','2026-10-02','Premium',1,1200,'Confirmed','App'),
(2003,'E03','Bhavna','2026-10-03','Standard',3,800,'Pending','Counter'),
(2004,'E04','Charan','2026-10-04','Standard',2,650,'Confirmed','Partner'),
(2005,'E05','Diya','2026-10-05','VIP',1,1500,'Cancelled','Web'),
(2006,'E06','Eshan','2026-10-06','Standard',4,400,'Confirmed','App'),
(2007,'E01','Fiza','2026-10-07','Premium',1,1000,'Confirmed','App'),
(2008,'E02','Farah','2026-10-08','Premium',2,1200,'Pending','Web'),
(2009,'E03','Gauri','2026-10-09','Standard',1,800,'Confirmed','Counter'),
(2010,'E04','Harini','2026-10-10','Standard',3,650,'Confirmed','Web'),
(2011,'E05','Irfan','2026-10-11','VIP',2,1500,'Confirmed','Partner'),
(2012,'E06','Jiya','2026-10-12','Standard',1,400,'Cancelled','App'),
(2013,'E01','Kabir','2026-10-13','Standard',4,500,'Confirmed','Web'),
(2014,'E02','Lakshmi','2026-10-14','VIP',1,1800,'Confirmed','Counter'),
(2015,'E03','Mohan','2026-10-15','Standard',2,800,'Confirmed','Partner'),
(2016,'E04','Nisha','2026-10-16','Standard',1,650,'Pending','Web'),
(2017,'E05','Omkar','2026-10-17','VIP',3,1500,'Confirmed','App'),
(2018,'E06','Pari','2026-10-18','Standard',2,400,'Confirmed','Web'),
(2019,'E01','Qadir','2026-10-19','Standard',2,500,'Cancelled','Counter'),
(2020,'E02','Rohan','2026-10-20','Premium',3,1200,'Confirmed','Partner'),
(2021,'E03','Sana','2026-10-21','Premium',1,1300,'Confirmed','App'),
(2022,'E04','Tanvi','2026-10-22','Standard',2,650,'Confirmed',NULL),
(2023,'E05','Uday','2026-10-23','VIP',1,1500,'Pending','Web'),
(2024,'E06','Imran','2026-10-24','Standard',3,400,'Confirmed','App'),
(2025,'E01','Varsha','2026-10-25','Standard',1,500,'Confirmed','Web'),
(2026,'E02','Wasim','2026-10-26','Premium',2,1200,'Confirmed',NULL),
(2027,'E03','Yash','2026-10-27','Standard',0,800,'Pending','Counter'),
(2028,'E04','Zara','2026-10-28','Standard',2,0,'Confirmed','Web'),
(2029,'E05','Adil','2026-10-29','VIP',2,1500,'Confirmed','App'),
(2030,'E06','Bina','2026-10-30','Standard',1,400,'Pending','Counter'),
(2031,'E99','Chetan','2026-10-31','Standard',2,700,'Confirmed','Web'),
(2032,'E01','Danish','2026-11-01','Standard',3,500,'Confirmed','Partner');

num_affected_rows,num_inserted_rows
32,32


In [0]:
%sql
CREATE OR REPLACE TABLE event_master (
  event_id STRING, event_name STRING, city STRING, category STRING, capacity INT
) USING DELTA;

In [0]:
%sql
INSERT INTO event_master VALUES
('E01','Design Summit','Bengaluru','Technology',300),
('E02','Food Expo','Hyderabad','Culinary',250),
('E03','Art Fair','Pune','Culture',180),
('E04','Startup Meetup','Mumbai','Business',220),
('E05','Music Showcase','Chennai','Entertainment',400),
('E06','Book Festival','Delhi','Literature',150),
('E07','Photography Walk','Jaipur','Culture',90);

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
CREATE OR REPLACE TABLE event_registrations_work USING DELTA
AS SELECT * FROM event_registrations_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_registrations AS
SELECT * FROM VALUES
(2002,'E02','Arnav','2026-10-02','Premium',1,1400,'Confirmed','App'),
(2008,'E02','Farah','2026-10-08','Premium',2,1200,'Confirmed','Web'),
(2024,'E06','Imran','2026-10-24','Standard',3,450,'Confirmed','App'),
(2033,'E07','Naina','2026-11-05','Standard',2,600,'Confirmed','Web'),
(2034,'E03','Omar','2026-11-05','Premium',1,900,'Pending','Partner'),
(2035,'E05','Leena','2026-11-06','VIP',2,1600,'Confirmed','App')
AS t(registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel);

In [0]:
%sql
-- Volumes used later for Parquet (Q45) and streaming checkpoints (Section I)
CREATE VOLUME IF NOT EXISTS databricks_rr1.default.event_files;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS databricks_rr1.default.checkpoints;

In [0]:
%sql
-- Q1
SELECT * FROM event_registrations_source;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2003,E03,Bhavna,2026-10-03,Standard,3,800,Pending,Counter
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner
2005,E05,Diya,2026-10-05,VIP,1,1500,Cancelled,Web
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web


In [0]:
%sql
-- Q2
SELECT registration_id, attendee_name, event_id, unit_price FROM event_registrations_source;

registration_id,attendee_name,event_id,unit_price
2001,Aarav,E01,500
2002,Arnav,E02,1200
2003,Bhavna,E03,800
2004,Charan,E04,650
2005,Diya,E05,1500
2006,Eshan,E06,400
2007,Fiza,E01,1000
2008,Farah,E02,1200
2009,Gauri,E03,800
2010,Harini,E04,650


In [0]:
%sql
-- Q3
SELECT * FROM event_registrations_source WHERE status = 'Confirmed';

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web
2011,E05,Irfan,2026-10-11,VIP,2,1500,Confirmed,Partner
2013,E01,Kabir,2026-10-13,Standard,4,500,Confirmed,Web
2014,E02,Lakshmi,2026-10-14,VIP,1,1800,Confirmed,Counter


In [0]:
%sql
-- Q4
SELECT * FROM event_registrations_source WHERE unit_price > 1000;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2005,E05,Diya,2026-10-05,VIP,1,1500,Cancelled,Web
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2011,E05,Irfan,2026-10-11,VIP,2,1500,Confirmed,Partner
2014,E02,Lakshmi,2026-10-14,VIP,1,1800,Confirmed,Counter
2017,E05,Omkar,2026-10-17,VIP,3,1500,Confirmed,App
2020,E02,Rohan,2026-10-20,Premium,3,1200,Confirmed,Partner
2021,E03,Sana,2026-10-21,Premium,1,1300,Confirmed,App
2023,E05,Uday,2026-10-23,VIP,1,1500,Pending,Web
2026,E02,Wasim,2026-10-26,Premium,2,1200,Confirmed,null


In [0]:
%sql
-- Q5
SELECT * FROM event_registrations_source WHERE ticket_type IN ('Premium','VIP');

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2005,E05,Diya,2026-10-05,VIP,1,1500,Cancelled,Web
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2011,E05,Irfan,2026-10-11,VIP,2,1500,Confirmed,Partner
2014,E02,Lakshmi,2026-10-14,VIP,1,1800,Confirmed,Counter
2017,E05,Omkar,2026-10-17,VIP,3,1500,Confirmed,App
2020,E02,Rohan,2026-10-20,Premium,3,1200,Confirmed,Partner
2021,E03,Sana,2026-10-21,Premium,1,1300,Confirmed,App
2023,E05,Uday,2026-10-23,VIP,1,1500,Pending,Web


In [0]:
%sql
-- Q6
SELECT * FROM event_registrations_source
WHERE unit_price BETWEEN 500 AND 1200
ORDER BY unit_price DESC;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2020,E02,Rohan,2026-10-20,Premium,3,1200,Confirmed,Partner
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2026,E02,Wasim,2026-10-26,Premium,2,1200,Confirmed,null
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2003,E03,Bhavna,2026-10-03,Standard,3,800,Pending,Counter
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter
2015,E03,Mohan,2026-10-15,Standard,2,800,Confirmed,Partner
2027,E03,Yash,2026-10-27,Standard,0,800,Pending,Counter
2031,E99,Chetan,2026-10-31,Standard,2,700,Confirmed,Web


In [0]:
%sql
-- Q7
SELECT DISTINCT channel FROM event_registrations_source;

channel
Web
App
Counter
Partner
null


In [0]:
%sql
-- Q8
SELECT * FROM event_registrations_source
WHERE attendee_name LIKE 'A%' OR attendee_name LIKE '%a%';

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2003,E03,Bhavna,2026-10-03,Standard,3,800,Pending,Counter
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner
2005,E05,Diya,2026-10-05,VIP,1,1500,Cancelled,Web
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web


In [0]:
%sql
-- Q9
SELECT event_id, SUM(tickets) AS total_tickets
FROM event_registrations_source
GROUP BY event_id ORDER BY event_id;

event_id,total_tickets
E01,13
E02,9
E03,7
E04,10
E05,9
E06,11
E99,2


In [0]:
%sql
-- Q10
SELECT event_id, SUM(tickets * unit_price) AS total_revenue
FROM event_registrations_source
WHERE status = 'Confirmed' AND tickets > 0 AND unit_price > 0
GROUP BY event_id ORDER BY event_id;

event_id,total_revenue
E01,6000
E02,9000
E03,3700
E04,4550
E05,10500
E06,3600
E99,1400


In [0]:
%sql
-- Q11
SELECT r.attendee_name, m.event_name, m.city, r.ticket_type, r.status
FROM event_registrations_source r
JOIN event_master m ON r.event_id = m.event_id;

attendee_name,event_name,city,ticket_type,status
Aarav,Design Summit,Bengaluru,Standard,Confirmed
Arnav,Food Expo,Hyderabad,Premium,Confirmed
Bhavna,Art Fair,Pune,Standard,Pending
Charan,Startup Meetup,Mumbai,Standard,Confirmed
Diya,Music Showcase,Chennai,VIP,Cancelled
Eshan,Book Festival,Delhi,Standard,Confirmed
Fiza,Design Summit,Bengaluru,Premium,Confirmed
Farah,Food Expo,Hyderabad,Premium,Pending
Gauri,Art Fair,Pune,Standard,Confirmed
Harini,Startup Meetup,Mumbai,Standard,Confirmed


In [0]:
%sql
-- Q12a: registrations whose event_id does not exist in event_master (expect E99)
SELECT r.*
FROM event_registrations_source r
LEFT JOIN event_master m ON r.event_id = m.event_id
WHERE m.event_id IS NULL;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2031,E99,Chetan,2026-10-31,Standard,2,700,Confirmed,Web


In [0]:
%sql
-- Q12b: events with no registrations (expect E07)
SELECT m.*
FROM event_master m
LEFT JOIN event_registrations_source r ON m.event_id = r.event_id
WHERE r.event_id IS NULL;

event_id,event_name,city,category,capacity
E07,Photography Walk,Jaipur,Culture,90


In [0]:
# Q13
reg_df = spark.read.table("event_registrations_source")
evt_df = spark.read.table("event_master")
reg_df.printSchema()
evt_df.printSchema()

root
 |-- registration_id: integer (nullable = true)
 |-- event_id: string (nullable = true)
 |-- attendee_name: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- ticket_type: string (nullable = true)
 |-- tickets: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- capacity: integer (nullable = true)



In [0]:
# Q14
display(reg_df.select("registration_id", "attendee_name", "tickets", "unit_price"))

registration_id,attendee_name,tickets,unit_price
2001,Aarav,2,500
2002,Arnav,1,1200
2003,Bhavna,3,800
2004,Charan,2,650
2005,Diya,1,1500
2006,Eshan,4,400
2007,Fiza,1,1000
2008,Farah,2,1200
2009,Gauri,1,800
2010,Harini,3,650


In [0]:
# Q15
from pyspark.sql.functions import col, when, sum as _sum
display(reg_df.filter((col("status") == "Confirmed") & (col("tickets") > 2)))

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web
2013,E01,Kabir,2026-10-13,Standard,4,500,Confirmed,Web
2017,E05,Omkar,2026-10-17,VIP,3,1500,Confirmed,App
2020,E02,Rohan,2026-10-20,Premium,3,1200,Confirmed,Partner
2024,E06,Imran,2026-10-24,Standard,3,400,Confirmed,App
2032,E01,Danish,2026-11-01,Standard,3,500,Confirmed,Partner


In [0]:
# Q16
reg_df2 = reg_df.withColumn("total_amount", col("tickets") * col("unit_price"))
display(reg_df2)

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel,total_amount
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web,1000
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App,1200
2003,E03,Bhavna,2026-10-03,Standard,3,800,Pending,Counter,2400
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner,1300
2005,E05,Diya,2026-10-05,VIP,1,1500,Cancelled,Web,1500
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App,1600
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App,1000
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web,2400
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter,800
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web,1950


In [0]:
# Q17
reg_df3 = reg_df2.withColumn("booking_category", when(col("total_amount") >= 2000, "High").otherwise("Regular"))
display(reg_df3)

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel,total_amount,booking_category
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web,1000,Regular
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App,1200,Regular
2003,E03,Bhavna,2026-10-03,Standard,3,800,Pending,Counter,2400,High
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner,1300,Regular
2005,E05,Diya,2026-10-05,VIP,1,1500,Cancelled,Web,1500,Regular
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App,1600,Regular
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App,1000,Regular
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web,2400,High
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter,800,Regular
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web,1950,Regular


In [0]:
# Q18
joined = reg_df2.join(evt_df, "event_id")
result = (joined.filter(col("status") == "Confirmed")
                .groupBy("city")
                .agg(_sum("total_amount").alias("total_confirmed_revenue")))
display(result)

city,total_confirmed_revenue
Delhi,3600
Chennai,10500
Hyderabad,9000
Pune,3700
Mumbai,4550
Bengaluru,6000


In [0]:
%sql
-- Q19
CREATE OR REPLACE TABLE confirmed_event_registrations USING DELTA
AS SELECT * FROM event_registrations_source WHERE status = 'Confirmed';

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM confirmed_event_registrations;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web
2011,E05,Irfan,2026-10-11,VIP,2,1500,Confirmed,Partner
2013,E01,Kabir,2026-10-13,Standard,4,500,Confirmed,Web
2014,E02,Lakshmi,2026-10-14,VIP,1,1800,Confirmed,Counter


In [0]:
# Q20
pending_df = spark.read.table("event_registrations_source").filter("status = 'Pending'")
pending_df.write.format("delta").mode("overwrite").saveAsTable("pending_event_registrations")
display(spark.read.table("pending_event_registrations"))

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2003,E03,Bhavna,2026-10-03,Standard,3,800,Pending,Counter
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2016,E04,Nisha,2026-10-16,Standard,1,650,Pending,Web
2023,E05,Uday,2026-10-23,VIP,1,1500,Pending,Web
2027,E03,Yash,2026-10-27,Standard,0,800,Pending,Counter
2030,E06,Bina,2026-10-30,Standard,1,400,Pending,Counter


In [0]:
%sql
-- Q21
INSERT INTO event_registrations_work VALUES
(2036,'E03','Meera','2026-11-02','Standard',2,800,'Confirmed','Web'),
(2037,'E04','Nikhil','2026-11-03','Premium',1,1100,'Pending','App');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
-- Q22
UPDATE event_registrations_work SET status = 'Confirmed' WHERE registration_id = 2003;

num_affected_rows
1


In [0]:
%sql
SELECT * FROM event_registrations_work WHERE registration_id = 2003;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2003,E03,Bhavna,2026-10-03,Standard,3,800,Confirmed,Counter


In [0]:
%sql
-- Q23
UPDATE event_registrations_work SET unit_price = unit_price + 200 WHERE ticket_type = 'VIP';

num_affected_rows
6


In [0]:
%sql
SELECT * FROM event_registrations_work WHERE ticket_type = 'VIP';

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2005,E05,Diya,2026-10-05,VIP,1,1700,Cancelled,Web
2011,E05,Irfan,2026-10-11,VIP,2,1700,Confirmed,Partner
2014,E02,Lakshmi,2026-10-14,VIP,1,2000,Confirmed,Counter
2017,E05,Omkar,2026-10-17,VIP,3,1700,Confirmed,App
2023,E05,Uday,2026-10-23,VIP,1,1700,Pending,Web
2029,E05,Adil,2026-10-29,VIP,2,1700,Confirmed,App


In [0]:
%sql
-- Q24
DELETE FROM event_registrations_work WHERE status = 'Cancelled';

num_affected_rows
3


In [0]:
%sql
SELECT * FROM event_registrations_work ORDER BY registration_id;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2003,E03,Bhavna,2026-10-03,Standard,3,800,Confirmed,Counter
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web
2011,E05,Irfan,2026-10-11,VIP,2,1700,Confirmed,Partner


In [0]:
%sql
-- Q25: existing IDs
SELECT i.registration_id, 'EXISTING' AS record_type
FROM incoming_registrations i
JOIN event_registrations_work w ON i.registration_id = w.registration_id;

registration_id,record_type
2002,EXISTING
2008,EXISTING
2024,EXISTING


In [0]:
%sql
-- Q25: new IDs
SELECT i.registration_id, 'NEW' AS record_type
FROM incoming_registrations i
LEFT ANTI JOIN event_registrations_work w ON i.registration_id = w.registration_id;

registration_id,record_type
2033,NEW
2034,NEW
2035,NEW


In [0]:
%sql
-- Q26
MERGE INTO event_registrations_work AS t
USING incoming_registrations AS s
ON t.registration_id = s.registration_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,3,0,3


In [0]:
%sql
-- Q27
SELECT * FROM event_registrations_work WHERE registration_id IN (2002, 2008, 2024);

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2002,E02,Arnav,2026-10-02,Premium,1,1400,Confirmed,App
2008,E02,Farah,2026-10-08,Premium,2,1200,Confirmed,Web
2024,E06,Imran,2026-10-24,Standard,3,450,Confirmed,App


In [0]:
%sql
-- Q28
SELECT * FROM event_registrations_work WHERE registration_id IN (2033, 2034, 2035);

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2033,E07,Naina,2026-11-05,Standard,2,600,Confirmed,Web
2034,E03,Omar,2026-11-05,Premium,1,900,Pending,Partner
2035,E05,Leena,2026-11-06,VIP,2,1600,Confirmed,App


## Section F - Delta Lake internals and Time Travel
(The PDF numbering jumps from Q28 to Q31.)

In [0]:
%sql
-- Q31: read format, location, numFiles and sizeInBytes from the output
DESCRIBE DETAIL event_registrations_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,182b9ce1-a4af-48d3-9641-0a3a9e181979,databricks_rr1.default.event_registrations_work,null,abfss://unity-catalog-storage@dbstoragezi3amqsua7fge.dfs.core.windows.net/7405610695820632/__unitystorage/catalogs/c47ff2ca-0659-4a6a-9cb1-459998e4b0dd/tables/bc5b0ff4-7046-4d29-a5a6-2caf06a1c541,2026-11-06T06:24:46.449Z,2026-11-06T06:37:07.000Z,List(),List(),1,3448,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Q32: see the operation column (CREATE TABLE AS SELECT, WRITE, UPDATE, DELETE, MERGE)
DESCRIBE HISTORY event_registrations_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-11-06T06:37:07.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2857903076041945),56fed703-7971-49bc-a610-8dfbfe57433c,1106-041906-napwor9s-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 7, numRemovedBytes -> 18794, p25FileSize -> 3448, numDeletionVectorsRemoved -> 1, minFileSize -> 3448, numAddedFiles -> 1, maxFileSize -> 3448, p75FileSize -> 3448, p50FileSize -> 3448, numAddedBytes -> 3448)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-06T06:37:05.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(registration_id#23182 = registration_id#23155)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2857903076041945),56fed703-7971-49bc-a610-8dfbfe57433c,1106-041906-napwor9s-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 6, numTargetBytesAdded -> 15347, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 2690, materializeSourceTimeMs -> 99, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 850, numTargetRowsUpdated -> 3, numOutputRows -> 6, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1717)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T06:34:03.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(status#22486 = Cancelled)""])",null,List(2857903076041945),448a43b9-6dcf-4b9a-8501-169fb8e202a4,1106-041906-napwor9s-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1060, numDeletionVectorsUpdated -> 0, numDeletedRows -> 3, scanTimeMs -> 664, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 395)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T06:33:45.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2857903076041945),eecc2bd9-0eda-4002-99ef-8128cf20797f,1106-041906-napwor9s-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 11618, p25FileSize -> 3447, numDeletionVectorsRemoved -> 1, minFileSize -> 3447, numAddedFiles -> 1, maxFileSize -> 3447, p75FileSize -> 3447, p50FileSize -> 3447, numAddedBytes -> 3447)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T06:33:43.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(ticket_type#21560 = VIP)""])",null,List(2857903076041945),eecc2bd9-0eda-4002-99ef-8128cf20797f,1106-041906-napwor9s-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1743, numDeletionVectorsUpdated -> 1, scanTimeMs -> 510, numAddedFiles -> 1, numUpdatedRows -> 6, numAddedBytes -> 2945, rewriteTimeMs -> 1233)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T06:33:23.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Ma

In [0]:
# Q33
hist = spark.sql("DESCRIBE HISTORY event_registrations_work")
merge_v = hist.filter("operation = 'MERGE'").agg({"version": "max"}).collect()[0][0]
before_v = merge_v - 1
print("MERGE version:", merge_v, "| version before MERGE:", before_v)
display(spark.sql(f"SELECT * FROM event_registrations_work VERSION AS OF {before_v} ORDER BY registration_id"))

MERGE version: 6 | version before MERGE: 5


registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2001,E01,Aarav,2026-10-01,Standard,2,500,Confirmed,Web
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App
2003,E03,Bhavna,2026-10-03,Standard,3,800,Confirmed,Counter
2004,E04,Charan,2026-10-04,Standard,2,650,Confirmed,Partner
2006,E06,Eshan,2026-10-06,Standard,4,400,Confirmed,App
2007,E01,Fiza,2026-10-07,Premium,1,1000,Confirmed,App
2008,E02,Farah,2026-10-08,Premium,2,1200,Pending,Web
2009,E03,Gauri,2026-10-09,Standard,1,800,Confirmed,Counter
2010,E04,Harini,2026-10-10,Standard,3,650,Confirmed,Web
2011,E05,Irfan,2026-10-11,VIP,2,1700,Confirmed,Partner


In [0]:
# Q34
print("BEFORE merge (historical version):")
display(spark.sql(f"SELECT * FROM event_registrations_work VERSION AS OF {before_v} WHERE registration_id = 2002"))
print("CURRENT record:")
display(spark.sql("SELECT * FROM event_registrations_work WHERE registration_id = 2002"))

BEFORE merge (historical version):


registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2002,E02,Arnav,2026-10-02,Premium,1,1200,Confirmed,App


CURRENT record:


registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2002,E02,Arnav,2026-10-02,Premium,1,1400,Confirmed,App


In [0]:
%sql
-- Q36
VACUUM event_registrations_work DRY RUN;

path


In [0]:
%sql
-- Q37: check the "Type" row (MANAGED / EXTERNAL) and the "Location" row
DESCRIBE EXTENDED event_registrations_source;

col_name,data_type,comment
registration_id,int,null
event_id,string,null
attendee_name,string,null
registration_date,string,null
ticket_type,string,null
tickets,int,null
unit_price,int,null
status,string,null
channel,string,null
,,


In [0]:
%sql
-- Q38
CREATE OR REPLACE VIEW event_revenue_view AS
SELECT m.event_name, m.city, SUM(r.tickets * r.unit_price) AS total_confirmed_revenue
FROM event_registrations_work r
JOIN event_master m ON r.event_id = m.event_id
WHERE r.status = 'Confirmed'
GROUP BY m.event_name, m.city;

In [0]:
%sql
-- Q39
CREATE OR REPLACE TEMP VIEW pending_registrations_tmp AS
SELECT * FROM event_registrations_work WHERE status = 'Pending';

In [0]:
%sql
-- Q40 (needs a classic cluster; not supported on serverless)
CREATE OR REPLACE TEMP VIEW vip_registrations_gtmp AS
SELECT * FROM event_registrations_work WHERE ticket_type = 'VIP';

In [0]:
%sql
-- Q41: permanent view
SELECT * FROM event_revenue_view;

event_name,city,total_confirmed_revenue
Startup Meetup,Mumbai,4550
Photography Walk,Jaipur,1200
Design Summit,Bengaluru,6000
Music Showcase,Chennai,15100
Book Festival,Delhi,3750
Food Expo,Hyderabad,11800
Art Fair,Pune,7700


In [0]:
%sql
-- Q41: temporary view
SELECT * FROM pending_registrations_tmp;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2023,E05,Uday,2026-10-23,VIP,1,1700,Pending,Web
2016,E04,Nisha,2026-10-16,Standard,1,650,Pending,Web
2027,E03,Yash,2026-10-27,Standard,0,800,Pending,Counter
2030,E06,Bina,2026-10-30,Standard,1,400,Pending,Counter
2037,E04,Nikhil,2026-11-03,Premium,1,1100,Pending,App
2034,E03,Omar,2026-11-05,Premium,1,900,Pending,Partner


In [0]:
%sql
-- Q41: global temporary view
SELECT * FROM vip_registrations_gtmp;

registration_id,event_id,attendee_name,registration_date,ticket_type,tickets,unit_price,status,channel
2011,E05,Irfan,2026-10-11,VIP,2,1700,Confirmed,Partner
2014,E02,Lakshmi,2026-10-14,VIP,1,2000,Confirmed,Counter
2017,E05,Omkar,2026-10-17,VIP,3,1700,Confirmed,App
2023,E05,Uday,2026-10-23,VIP,1,1700,Pending,Web
2029,E05,Adil,2026-10-29,VIP,2,1700,Confirmed,App
2035,E05,Leena,2026-11-06,VIP,2,1600,Confirmed,App


In [0]:
%sql
-- Q42/Q43: file count BEFORE optimize
DESCRIBE DETAIL event_registrations_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,182b9ce1-a4af-48d3-9641-0a3a9e181979,databricks_rr1.default.event_registrations_work,null,abfss://unity-catalog-storage@dbstoragezi3amqsua7fge.dfs.core.windows.net/7405610695820632/__unitystorage/catalogs/c47ff2ca-0659-4a6a-9cb1-459998e4b0dd/tables/bc5b0ff4-7046-4d29-a5a6-2caf06a1c541,2026-11-06T06:24:46.449Z,2026-11-06T06:37:07.000Z,List(),List(),1,3448,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Q42
OPTIMIZE event_registrations_work;

path,metrics
abfss://unity-catalog-storage@dbstoragezi3amqsua7fge.dfs.core.windows.net/7405610695820632/__unitystorage/catalogs/c47ff2ca-0659-4a6a-9cb1-459998e4b0dd/tables/bc5b0ff4-7046-4d29-a5a6-2caf06a1c541,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793947503798, 1793947504458, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, null, null, 0)"


In [0]:
%sql
-- Q43: file count AFTER optimize
DESCRIBE DETAIL event_registrations_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,182b9ce1-a4af-48d3-9641-0a3a9e181979,databricks_rr1.default.event_registrations_work,null,abfss://unity-catalog-storage@dbstoragezi3amqsua7fge.dfs.core.windows.net/7405610695820632/__unitystorage/catalogs/c47ff2ca-0659-4a6a-9cb1-459998e4b0dd/tables/bc5b0ff4-7046-4d29-a5a6-2caf06a1c541,2026-11-06T06:24:46.449Z,2026-11-06T06:37:07.000Z,List(),List(),1,3448,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Q44
OPTIMIZE event_registrations_work ZORDER BY (event_id);

path,metrics
abfss://unity-catalog-storage@dbstoragezi3amqsua7fge.dfs.core.windows.net/7405610695820632/__unitystorage/catalogs/c47ff2ca-0659-4a6a-9cb1-459998e4b0dd/tables/bc5b0ff4-7046-4d29-a5a6-2caf06a1c541,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3448), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793947522884, 1793947523511, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, null, null, 0)"


In [0]:
# Q45
path = "/Volumes/databricks_rr1/default/event_files/event_master_parquet"

# 1. Write event records as Parquet
spark.read.table("event_master").write.mode("overwrite").parquet(path)

# 2. Read the Parquet data back
pq_df = spark.read.parquet(path)
display(pq_df)

# 3. Load into a Delta table
pq_df.write.format("delta").mode("overwrite").saveAsTable("event_master_delta")
display(spark.read.table("event_master_delta"))

event_id,event_name,city,category,capacity
E01,Design Summit,Bengaluru,Technology,300
E02,Food Expo,Hyderabad,Culinary,250
E03,Art Fair,Pune,Culture,180
E04,Startup Meetup,Mumbai,Business,220
E05,Music Showcase,Chennai,Entertainment,400
E06,Book Festival,Delhi,Literature,150
E07,Photography Walk,Jaipur,Culture,90


event_id,event_name,city,category,capacity
E01,Design Summit,Bengaluru,Technology,300
E02,Food Expo,Hyderabad,Culinary,250
E03,Art Fair,Pune,Culture,180
E04,Startup Meetup,Mumbai,Business,220
E05,Music Showcase,Chennai,Entertainment,400
E06,Book Festival,Delhi,Literature,150
E07,Photography Walk,Jaipur,Culture,90


In [0]:
# Q46
rate_df = (spark.readStream
           .format("rate")
           .option("rowsPerSecond", 5)
           .load())
print("Is streaming:", rate_df.isStreaming)

Is streaming: True


In [0]:
# Q47: columns are timestamp (event time generated by the source) and value (a long counter starting at 0)
rate_df.printSchema()
display(rate_df, checkpointLocation="/Volumes/databricks_rr1/default/checkpoints/e47")

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)

Checkpointing to /Volumes/databricks_rr1/default/checkpoints/e47


In [0]:
# Q48
from pyspark.sql.functions import col, concat, lit

txn_df = (rate_df
          .withColumn("transaction_id", concat(lit("TXN-"), col("value").cast("string")))
          .withColumn("transaction_amount", (col("value") % 20) * 50))
display(txn_df, checkpointLocation="/Volumes/databricks_rr1/default/checkpoints/e48")

Checkpointing to /Volumes/databricks_rr1/default/checkpoints/e48


In [0]:
# Q49
high_df = txn_df.filter(col("transaction_amount") > 500)
display(high_df, checkpointLocation="/Volumes/databricks_rr1/default/checkpoints/e49")

Checkpointing to /Volumes/databricks_rr1/default/checkpoints/e49


In [0]:
# Q50
cat_df = txn_df.withColumn(
    "txn_category",
    when(col("transaction_amount") >= 800, "High")
    .when(col("transaction_amount") >= 400, "Medium")
    .otherwise("Low")
)
agg_df = cat_df.groupBy("txn_category").count()
display(agg_df, checkpointLocation="/Volumes/databricks_rr1/default/checkpoints/e50")

Checkpointing to /Volumes/databricks_rr1/default/checkpoints/e50


In [0]:
# Q50: stop all running streaming queries
for s in spark.streams.active:
    print("Stopping:", s.id)
    s.stop()